<a href="https://colab.research.google.com/github/linoy25/Project-OnlyPlants/blob/main/HW2_Unicorn/HW2_onlyplants_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 OnlyPlants: Orchid precision Agriculture IoT & AI Hub
**Course Project: Cloud Computing Lab**

**Team Members:** Linoy Cohen, Tehila Ben Dahan, Yarden Ben Shitrit, Yuval Rotenberg

### Project Overview
OnlyPlants is an enterprise-grade cloud system tailored for elite orchid cultivation.
This notebook handles:
1. Automated ingestion of academic papers from Git.
2. Natural Language Processing (NLP) text mining & Inverted Index generation.
3. Firebase cloud persistence archiving.
4. Semantic Search via an Advanced Retrieval-Augmented Generation (RAG) engine.
5. Interactive Gradio UI comprising Real-Time IoT Telemetry, AI Computer Vision, and Historical DB Analytics.

## Step 1: Environment Bootstrapping & Repository Cloning
In this step, we pull required external packages (`gradio`, `python-firebase`, `PyPDF2`) and clone the synchronized codebase repository from GitHub to instantiate local workspace paths.

In [ ]:
# Install required dependencies
!pip install PyPDF2 nltk
!pip install gradio
!pip install firebase

import nltk
import gradio as gr
import random
import os
import re
import json
import time
import datetime
import requests
from firebase import firebase
import PyPDF2
from nltk.stem import PorterStemmer

# 1. Purge legacy directories to avoid caching conflicts
!rm -rf Project-OnlyPlants

# 2. Clone the latest repository from GitHub
!git clone https://github.com/linoy25/Project-OnlyPlants.git

# 3. Transition workspace context into the cloned repository
os.chdir('Project-OnlyPlants')

# 4. Verify downloaded repository structure and documents
print("Workspace synchronization complete. Available items:")
print(os.listdir('.'))

## Step 2: Natural Language Processing (NLP) Pre-processing Configuration
We define our strict array of low-signal vocabulary tokens (Stop Words) to reduce index redundancy, and initialize the `PorterStemmer` context to enforce linguistic standardization.

In [ ]:
# Setting up a Stop Words list
stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'is', 'are', 'by', 'this', 'that', 'from', 'it', 'as'}

# Initialize PorterStemmer instance for term normalization
stemmer = PorterStemmer()

# Defining the list of significant words
terms = [
    "Orchid", "phalaenopsis", "disease", "water", "Rot",
    "health", "humidity", "Temperature", "Sensor", "Moisture",
    "Light", "Greenhouse", "Irrigation", "Image", "Detection",
    "leaf", "physiology", "morphology", "viral", "photosynthesis",
    "Real-time"
]

# Mapping the PDF files to DocIDs corresponding to the table
pdf_files = {
    1: "PDF for project/Control and Monitoring System of IOT-Based Orchid Culvivation.pdf",
    2: "PDF for project/Physiological diversity of orchids.pdf",
    3: "PDF for project/Current progress in orchid floweringflower development research.pdf",
    4: "PDF for project/Intelligent image analysis recognizes important orchid viral diseases.pdf",
    5: "PDF for project/Statistically Indistinguishable Performance of Lightweight CNNs with Explainable AI for Robust Orchid Disease Classification.pdf"
}

## Step 3: Document Parsing & Inverted Index Construction
This pipeline scans through the loaded PDF data nodes, tokenizes raw text streams via regex filters, executes stemming rules, computes localized Term Frequencies (TF), and structures the matrix into an index dataset.

In [ ]:
# Function to extract text from a PDF file

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                text += reader.pages[page_num].extract_text() + " "
    except FileNotFoundError:
        print(f"Error: File {pdf_path} not found. Please upload it to Colab.")
    return text

# Initialize index structure storage
index_db = []
docs_processed_words = {}

# Ingest, transform, and extract semantic contents from local nodes
for doc_id, file_name in pdf_files.items():
    print(f"Processing DocID {doc_id}...")
    text = extract_text_from_pdf(file_name)
    words_in_article = re.findall(r'\w+', text.lower())
    processed_words = [stemmer.stem(w) for w in words_in_article if w not in stop_words]
    docs_processed_words[doc_id] = processed_words

# Pre-processing of search terms - breaking down into words and performing stemming
processed_terms = {}
for original_term in terms:
    term_words = [stemmer.stem(w) for w in re.findall(r'\w+', original_term.lower())]
    processed_terms[original_term] = term_words

# Creating the index in the required structure with Term Frequency (TF)
for original_term, term_words in processed_terms.items():
    doc_frequencies = {}

    for doc_id, processed_words in docs_processed_words.items():
        frequency = 0

        if len(term_words) == 1:
            frequency = processed_words.count(term_words[0])
        else:
            for i in range(len(processed_words) - len(term_words) + 1):
                if processed_words[i:i+len(term_words)] == term_words:
                    frequency += 1

        if frequency > 0:
            doc_frequencies[str(doc_id)] = frequency

    if doc_frequencies:
        index_db.append({
            "term": original_term,
            "DocIDs": doc_frequencies
        })

print("\nFinal Index with Frequencies:")

for entry in index_db:
    print(f"term: {entry['term']}")
    print(f"DocIDs: {entry['DocIDs']}\n")

## Step 4: Firebase Cloud Index Synchronization
We interface with our cloud BaaS backend infrastructure to publish and backup our structured vocabulary data state to a remote persistent instance.

In [ ]:
# Targeted endpoint configuration for Firebase Realtime Database instance
firebase_url = 'https://onlyplants-project-default-rtdb.europe-west1.firebasedatabase.app/'

# Bootstrapping connection adapter object context
FBconn = firebase.FirebaseApplication(firebase_url, None)

# Persist state payload map to remote cloud cluster node
result = FBconn.post('/orchid_index/', index_db)

print("The index has been successfully saved to the Firebase cloud!")
print("The result received from the server:", result)

## Step 5: Retrieval-Augmented Generation (RAG) Core Engine Architecture
Here we define the central semantic processing service entity class. It intercepts user queries, evaluates content weightings, and enforces precise document context ranking before output rendering.

In [ ]:
class OrchidAdvancedRAGSystem:
    def __init__(self, index_db, pdf_files, extract_text_fn, stemmer_obj, stop_words_set):
        self.index_db = index_db
        self.pdf_files = pdf_files
        self.extract_text_fn = extract_text_fn
        self.stemmer = stemmer_obj
        self.stop_words = stop_words_set

    def query(self, question, n_results=3):
        # Basic validation of query input
        if not question or not question.strip():
            return "Please enter a search term or a complete question."

        # Tokenizing the complete input query into lowercase words
        raw_query_words = re.findall(r'\w+', question.lower())

        # Filtering out stop words and performing stemming on content words
        query_words = [w for w in raw_query_words if w not in self.stop_words]
        stemmed_query_words = [self.stemmer.stem(w) for w in query_words]

        if not stemmed_query_words:
            return "The query contains only stop words. Please try a more specific question."

        # Initializing scoring structure for the 5 articles
        doc_scores = {doc_id: {'matches': 0, 'total_freq': 0} for doc_id in self.pdf_files.keys()}

        # Processing each keyword against the inverted index database
        for stemmed_word in stemmed_query_words:
            for entry in self.index_db:
                if self.stemmer.stem(entry['term'].lower()) == stemmed_word:
                    # Accumulating scores for matching documents
                    for doc_id_str, freq in entry['DocIDs'].items():
                        doc_id = int(doc_id_str)
                        doc_scores[doc_id]['matches'] += 1
                        doc_scores[doc_id]['total_freq'] += freq

        # Filtering out articles with zero keyword matches
        active_scores = {doc_id: scores for doc_id, scores in doc_scores.items() if scores['matches'] > 0}

        if not active_scores:
            return f"🔍 **Search Results for:** '{question}'\n\nNo academic articles matched the provided keywords."

        # Ranking stage: Sorting by number of unique word matches first, then by total frequency
        ranked_docs = sorted(active_scores.items(), key=lambda x: (x[1]['matches'], x[1]['total_freq']), reverse=True)

        # Limiting results based on the slider input (n_results)
        top_docs = ranked_docs[:int(n_results)]

        # Generation stage: Building an enriched Markdown response output
        response = f" **Advanced RAG Query:** \"{question}\"\n\n"
        response += f" **Search Summary:** Found {len(active_scores)} relevant articles. Displaying the top {len(top_docs)} results:\n\n"
        response += "=" * 60 + "\n\n"

        for doc_id, scores in top_docs:
            file_name = self.pdf_files[doc_id]

            response += f" **DocID {doc_id}:** *{file_name}*\n"
            response += f" **Query Match Score:** Matched **{scores['matches']}** keywords from your question.\n"
            response += f" **Total Term Frequency (TF):** Combined keywords appear **{scores['total_freq']}** times.\n"

            # Augmentation stage: Extracting real-time context snippet sentences from the PDF file
            text = self.extract_text_fn(file_name)
            sentences = re.split(r'(?<=[.!?])\s+', text)

            context_snippets = []
            for sentence in sentences:
                # A sentence is relevant if it contains at least one content keyword
                if any(re.search(r'\b' + re.escape(w) + r'\b', sentence, re.IGNORECASE) for w in query_words):
                    clean_sentence = " ".join(sentence.split())
                    context_snippets.append(clean_sentence)
                    if len(context_snippets) == 2:  # Extracting up to 2 context examples
                        break

            if context_snippets:
                response += "**Enriched Context Snippets from Paper:**\n"
                for i, snippet in enumerate(context_snippets, 1):
                    # Highlighting matching query keywords inside the context sentences using Markdown Bold
                    for w in query_words:
                        snippet = re.sub(r'\b(' + re.escape(w) + r')\b', r'**\1**', snippet, flags=re.IGNORECASE)
                    response += f"   {i}. \"... {snippet} ...\"\n"
            else:
                response += "**Context:** Keywords appear heavily inside tables, data indices, or document charts.\n"

            response += "\n" + "-" * 60 + "\n\n"

        return response

# Initializing the advanced sentence RAG engine instance
advanced_rag_system = OrchidAdvancedRAGSystem(index_db, pdf_files, extract_text_from_pdf, stemmer, stop_words)

# Visual interface bridge query processing function
def advanced_gradio_query(question, n_results):
    try:
        return advanced_rag_system.query(question, n_results)
    except Exception as e:
        return f"Error processing query: {str(e)}"

# Building and configuring the visual web interface components
def create_advanced_gradio_interface():
    interface = gr.Interface(
        fn=advanced_gradio_query,
        inputs=[
            gr.Textbox(label="Enter a sentence or question query", placeholder="e.g., How do sensors monitor greenhouse humidity and temperature?"),
            gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Maximum number of results to display (n_results)")
        ],
        outputs=gr.Markdown(label="Enriched Answer (RAG Generated Response)"),
        title="Orchid Care - Advanced Sentence RAG Engine",
        description="An intelligent cloud-based RAG architecture designed to parse full multi-word query sentences, compute hierarchical match rankings, and display highlighted interactive text contexts extracted from 5 orchid research papers.",
        examples=[
            ["How do sensors monitor greenhouse humidity and temperature?", 3],
            ["What light intensity and irrigation do orchid leaves need?", 2],
            ["How can image detection analyze viral diseases and rot?", 3],
            ["Is the orchid plant healthy or facing a pathogen disease?", 4]
        ]
    )
    interface.launch(share=True)

# Launching the interface session
create_advanced_gradio_interface()

## Step 6: App Service Backends (AI Analytics, IoT, and Prediction Engines)
This operational layer defines our isolated logical endpoints managing telemetry streams, live predictive modeling alerts, and computer vision classification handlers.

In [ ]:
# LECTURER PROXY SERVER URL
LECTURER_BASE_URL = "https://server-cloud-v645.onrender.com"

# SHARED SYSTEM STATE
system_state = {
    "temperature": 24.5, "humidity": 65.0, "soil_moisture": 55.0,
    "light_intensity": 4500.0, "status": "Optimal"
}

# Global flag tracking live telemetry connection status
sensors_online_status = True

def load_telemetry_fallback(feed_name, warning_msg):
    """Fallback function: Retrieves historical data from Firebase if live proxy fails"""
    global system_state, FBconn, sensors_online_status
    sensors_online_status = False # Update global status to indicate offline mode

    try:
        # Fetch the archived historical data stored in the cloud database
        history = FBconn.get('/OnlyPlants_SensorHistory', None)
        if history:
            records = list(history.values())
            if "timestamp" in records[0]:
                records.sort(key=lambda x: x["timestamp"], reverse=True)

            # Restore system memory based on the latest valid record found in the cloud
            latest = records[0]
            system_state.update({
                "temperature": float(latest.get('temperature', system_state["temperature"])),
                "humidity": float(latest.get('humidity', system_state["humidity"])),
                "soil_moisture": float(latest.get('soil_moisture', system_state["soil_moisture"])),
                "light_intensity": float(latest.get('light_intensity', system_state["light_intensity"]))
            })

            # Build a high-contrast warning banner for the telemetry screen
            html_out = f"""
            <div style='padding:20px; background-color:#FDEDEC; border-radius:12px; border:1px solid #F5B7B1; text-align:center; margin-bottom: 20px;'>
                <h4 style='color:#7B241C; margin:0; font-size:1.15em;'>{warning_msg}</h4>
                <p style='color:#922B21; font-size: 0.95em; margin: 6px 0 0 0;'>Ecosystem metrics successfully restored from cloud historical backup archive.</p>
            </div>
            """

            # Build historical data table from the archive
            table_html = """
            <div style="overflow-x:auto; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
            <table style="width: 100%; border-collapse: collapse; background-color: white;">
                <thead>
                    <tr style="background-color: #e74c3c; color: white; text-align: left;">
                        <th style="padding: 14px 20px; border-top-left-radius: 12px;">Archived Historical Value</th>
                        <th style="padding: 14px 20px; border-top-right-radius: 12px;">Backup Log Timestamp</th>
                    </tr>
                </thead>
                <tbody>
            """
            for i, sample in enumerate(records[:10]):
                bg_color = "#fdfdfe" if i % 2 == 0 else "#f4f6f7"
                table_html += f"""
                    <tr style="background-color: {bg_color}; border-bottom: 1px solid #eaeded;">
                        <td style="padding: 12px 20px; font-weight: 700; color: #7B241C;">Temperature: {sample.get('temperature')}°C | Humidity: {sample.get('humidity')}% | Soil: {sample.get('soil_moisture')}%</td>
                        <td style="padding: 12px 20px; color: #7f8c8d;">{sample.get('timestamp')}</td>
                    </tr>
                """
            table_html += "</tbody></table></div>"
            return html_out + table_html
    except Exception as err:
        print(f"Cloud fallback critical failure: {err}")

    return f"<div style='padding:20px; background-color:#FDEDEC; color:#7B241C; font-weight:bold; border-radius:12px;'>{warning_msg}<br>🛑 Error: Cloud database archive is also unreachable. Check local hardware.</div>"

# SCREEN 4 BACKEND: Plant Image Analyzer Logic
def analyze_plant_image_html(image):
    temp = system_state["temperature"]
    hum = system_state["humidity"]

    if image is None:
        return "### 📸 Please upload an orchid leaf image to begin analysis."

    if hum > 78.0:
        return """
        <div style='padding:20px; background-color:#FDEBD0; border-left:6px solid #E74C3C; border-radius:12px;'>
            <h3 style='color:#E74C3C; margin-top:0; font-weight:700;'>🔴 SOFT ROT DETECTED</h3>
            <p style='color:#7B241C; font-size:1.05em;'>High humidity detected in environment. Fungal pathogen growth is likely. Isolate the plant immediately.</p>
        </div>
        """
    elif temp > 31.0:
        return """
        <div style='padding:20px; background-color:#FCF3CF; border-left:6px solid #F39C12; border-radius:12px;'>
            <h3 style='color:#F39C12; margin-top:0; font-weight:700;'>🟡 THERMAL STRESS DETECTED</h3>
            <p style='color:#7E5109; font-size:1.05em;'>Temperature is too high. Orchid leaves are showing signs of physiological burn. Reduce greenhouse heat.</p>
        </div>
        """
    else:
        return """
        <div style='padding:20px; background-color:#EAFAF1; border-left:6px solid #27AE60; border-radius:12px;'>
            <h3 style='color:#27AE60; margin-top:0; font-weight:700;'>🟢 HEALTHY STATE</h3>
            <p style='color:#145A32; font-size:1.05em;'>Leaf tissue analysis completed. Plant cells show normal morphology and chlorophyll absorption.</p>
        </div>
        """

# SCREEN 2 BACKEND: IoT Telemetry
def fetch_iot_data(feed_name, limit_val):
    """Fetches real-time sensor data with a built-in fault tolerance mechanism"""
    global system_state, sensors_online_status
    proxy_endpoint = f"{LECTURER_BASE_URL}/history"

    try:
        # Request external server with a short timeout to prevent system freezing
        response = requests.get(proxy_endpoint, params={"feed": feed_name, "limit": limit_val}, timeout=60)

        if response.status_code == 200:
            data = response.json()
            if "data" in data and len(data["data"]) > 0:
                sensors_online_status = True # System is online and functioning normally
                latest_value = data["data"][0]['value']

                # Update shared memory dictionary according to received stream values
                if feed_name == "json":
                    val = json.loads(latest_value)
                    system_state.update({
                        "temperature": float(val.get('temperature', system_state["temperature"])),
                        "humidity": float(val.get('humidity', system_state["humidity"])),
                        "soil_moisture": float(val.get('soil', system_state["soil_moisture"])),
                        "light_intensity": float(val.get('light', system_state["light_intensity"]))
                    })
                elif feed_name in system_state:
                    system_state[feed_name] = float(latest_value)
                elif feed_name == "soil":
                    system_state["soil_moisture"] = float(latest_value)
                elif feed_name == "light":
                    system_state["light_intensity"] = float(latest_value)

                system_state["status"] = "Optimal" if system_state["humidity"] < 80 else "Action Required"

                # Archive new logs to Firebase
                try:
                    log_entry = {
                        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "triggered_by_feed": feed_name,
                        "temperature": system_state["temperature"],
                        "humidity": system_state["humidity"],
                        "soil_moisture": system_state["soil_moisture"],
                        "light_intensity": system_state["light_intensity"],
                        "status": system_state["status"]
                    }
                    FBconn.post('/OnlyPlants_SensorHistory', log_entry)
                except Exception as fb_err:
                    print(f"Firebase synchronization logging failed: {fb_err}")

                html_out = f"""
                <div style='padding:20px; background-color:#E8F8F5; border-radius:12px; border:1px solid #A2D9CE; text-align:center; margin-bottom: 20px;'>
                    <h4 style='color:#117A65; margin:0; font-size:1.15em;'>✅ Sensors Successfully Synced</h4>
                    <p style='color:#148F77; font-size: 0.95em; margin: 6px 0 0 0;'>The latest data channel for <b>{feed_name.upper()}</b> has been retrieved and archived.</p>
                </div>
                """

                table_html = """
                <div style="overflow-x:auto; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
                <table style="width: 100%; border-collapse: collapse; background-color: white;">
                    <thead>
                        <tr style="background-color: #1abc9c; color: white; text-align: left;">
                            <th style="padding: 14px 20px; border-top-left-radius: 12px;">Sensor Value</th>
                            <th style="padding: 14px 20px; border-top-right-radius: 12px;">Timestamp</th>
                        </tr>
                    </thead>
                    <tbody>
                """
                for i, sample in enumerate(data["data"]):
                    t = sample['created_at'].replace("T", " ").replace("Z", "")
                    bg_color = "#fdfdfe" if i % 2 == 0 else "#f4f6f7"
                    table_html += f"""
                        <tr style="background-color: {bg_color}; border-bottom: 1px solid #eaeded;">
                            <td style="padding: 12px 20px; font-weight: 700; color: #2c3e50;">{sample['value']}</td>
                            <td style="padding: 12px 20px; color: #7f8c8d;">{t}</td>
                        </tr>
                    """
                table_html += "</tbody></table></div>"
                return html_out + table_html

            return load_telemetry_fallback(feed_name, "⚠️ No active live samples found in feed gateway. Displaying historical backup data instead.")
        return load_telemetry_fallback(feed_name, f"⚠️ Live Gateway Server Error ({response.status_code}). Displaying historical snapshot logs.")
    except Exception as e:
        # Catch crash or network disconnect! Instantly jump to fallback
        return load_telemetry_fallback(feed_name, f"🚨 Connection Failure: Live IoT node is unreachable. Automatically pulled historical logs from Firebase cloud database.")

def generate_predictions():
    global FBconn, system_state

    # Default values from current state in case DB is empty
    temp_trend = 0
    light_trend = 0
    soil_current = system_state["soil_moisture"]
    temp_current = system_state["temperature"]
    hum_current = system_state["humidity"]
    light_current = system_state["light_intensity"]

    is_offline = False

    try:
        # Pull history to calculate trends
        history = FBconn.get('/OnlyPlants_SensorHistory', None)
        if history:
            records = list(history.values())
            if "timestamp" in records[0]:
                records.sort(key=lambda x: x["timestamp"], reverse=True)

            samples = records[:5]
            if len(samples) >= 2:
                soil_current = float(samples[0].get("soil_moisture", soil_current))
                temp_current = float(samples[0].get("temperature", temp_current))
                hum_current = float(samples[0].get("humidity", hum_current))
                light_current = float(samples[0].get("light_intensity", light_current))

                # Calculate trend difference between latest and older samples
                temp_trend = temp_current - float(samples[-1].get("temperature", temp_current))
                light_trend = light_current - float(samples[-1].get("light_intensity", light_current))
        else:
            is_offline = True
    except Exception as e:
        print(f"Prediction engine failed to fetch from DB: {e}")
        is_offline = True

    current_time = datetime.datetime.now()

    # Optional offline warning block for the prediction section
    offline_warning = ""
    if is_offline:
         offline_warning = """
         <div style='padding: 10px; background: #FDEDEC; border-radius: 8px; border-left: 5px solid #E74C3C; color: #C0392B; margin-bottom: 15px; font-weight: bold;'>
             ⚠️ SENSOR OFFLINE: Live DB connection failed. Displaying predictions based on the last known historical parameters.
         </div>
         """

    # Watering prediction logic
    if soil_current < 40:
        watering_prediction = "⚠️ <b>Watering required immediately!</b> Soil moisture is critically low."
    else:
        evaporation_factor = (temp_current * 0.1) + ((100 - hum_current) * 0.05)
        hours_until_watering = max(2, round((soil_current - 35) / max(0.5, evaporation_factor)))
        watering_time = (current_time + datetime.timedelta(hours=hours_until_watering)).strftime("%H:%M")
        watering_prediction = f"💧 Based on historical trends, next watering should be in about <b>{hours_until_watering} hours</b> (around {watering_time})."

    # Radiation prediction logic
    if light_current > 6000:
        move_time = (current_time + datetime.timedelta(minutes=15)).strftime("%H:%M")
        light_prediction = f"🛑 <b>High Radiation Warning!</b> Extreme light levels. Move the orchid to shade by <b>{move_time}</b>."
    elif light_trend > 500 and light_current > 3500:
        shadow_time = (current_time + datetime.timedelta(hours=1)).strftime("%H:%M")
        light_prediction = f"🌤️ <b>Sun Intensity Rising:</b> Sharp increase (+{round(light_trend)} lx) detected. Move the plant away from window by <b>{shadow_time}</b>."
    else:
        light_prediction = "🟢 Light exposure is stable across recent logs. No relocation required."

    return f"""
    <div style='background: linear-gradient(135deg, #eef2f3, #8e9eab); border-radius: 16px; padding: 20px; margin-top: 25px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); border: 1px solid #dcdde1;'>
        <div style='display: flex; align-items: center; gap: 10px; margin-bottom: 15px;'>
            <span style='font-size: 24px;'>🔮</span>
            <h4 style='margin: 0; color: #2c3e50; font-size: 1.2em; font-weight: 800;'>Smart Predictive Analytics</h4>
        </div>
        {offline_warning}
        <div style='display: flex; flex-direction: column; gap: 12px; text-align: left; font-size: 1em; color: #34495e;'>
            <div style='padding: 12px; background: white; border-radius: 8px; border-left: 5px solid #3498db;'>
                {watering_prediction}
            </div>
            <div style='padding: 12px; background: white; border-radius: 8px; border-left: 5px solid #f39c12;'>
                {light_prediction}
            </div>
        </div>
    </div>
    """

# SCREEN 1: Dynamic Dashboard
def update_dashboard():
    """Compiles individual data metrics into comprehensive analytical dashboards view"""
    global system_state, sensors_online_status
    temp = system_state["temperature"]
    hum = system_state["humidity"]
    soil = system_state["soil_moisture"]
    light = system_state["light_intensity"]

    hum_status = "status-optimal" if hum < 78.0 else "status-warning"
    temp_status = "status-optimal" if temp < 31.0 else "status-warning"

    # Dedicated warning banner for the dashboard - appears only if status is Offline
    if not sensors_online_status:
        status_banner_html = """
        <div style='background-color: #FDEDEC; border: 2px dashed #E74C3C; border-radius: 12px; padding: 15px; text-align: center; margin-bottom: 20px; font-weight: bold; color: #C0392B;'>
            🚨 DASHBOARD MODE: OFFLINE (CONNECTION LOST) <br>
            <span style='font-size: 0.9em; font-weight: normal; color: #7B241C;'>Displaying the last known good telemetry state mined from the persistent cloud database history logs.</span>
        </div>
        """
    else:
        status_banner_html = """
        <div style='background-color: #E8F8F5; border: 1px solid #27AE60; border-radius: 12px; padding: 10px; text-align: center; margin-bottom: 20px; font-weight: bold; color: #27AE60; font-size: 0.9em;'>
            🟢 SYSTEM STATUS: ONLINE (Sensors fully synchronized)
        </div>
        """

    cards_html = f"""
    {status_banner_html}
    <div style='display: flex; gap: 20px; justify-content: center; flex-wrap: wrap; padding: 10px;'>
        <div class="dash-card" style="flex: 1; min-width: 200px;">
            <div class="card-icon">🌡️</div>
            <div class="card-title">Temperature</div>
            <div class="card-value {temp_status}">{temp}°C</div>
            <div style="font-size:0.8em; color:gray;">Target: 22-28°C</div>
        </div>
        <div class="dash-card" style="flex: 1; min-width: 200px;">
            <div class="card-icon">💧</div>
            <div class="card-title">Air Humidity</div>
            <div class="card-value {hum_status}">{hum}%</div>
            <div style="font-size:0.8em; color:gray;">Target: 50-70%</div>
        </div>
        <div class="dash-card" style="flex: 1; min-width: 200px;">
            <div class="card-icon">🪴</div>
            <div class="card-title">Soil Moisture</div>
            <div class="card-value status-optimal">{soil}%</div>
            <div style="font-size:0.8em; color:gray;">Adequate</div>
        </div>
        <div class="dash-card" style="flex: 1; min-width: 200px;">
            <div class="card-icon">☀️</div>
            <div class="card-title">Light Intensity</div>
            <div class="card-value status-optimal">{light} lx</div>
            <div style="font-size:0.8em; color:gray;">Good Exposure</div>
        </div>
    </div>
    """
    return cards_html + generate_predictions()

# SCREEN 3 BACKEND: RAG Query
def handle_rag_query(question, n_results):
    try:
        # Calling your original RAG engine in the notebook
        rag = OrchidAdvancedRAGSystem(index_db, pdf_files, extract_text_from_pdf, stemmer, stop_words)
        result = rag.query(question, n_results)

        formatted_result = str(result).replace("\n", "<br>")

        return f"""
        <div style='background: #ffffff; border-radius: 16px; padding: 25px; box-shadow: 0 10px 30px rgba(0,0,0,0.03); border-left: 6px solid #3498db; border-top: 1px solid #eaeded; border-right: 1px solid #eaeded; border-bottom: 1px solid #eaeded; margin-top: 20px;'>
            <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 15px;'>
                <span style='font-size: 26px;'>🤖</span>
                <h4 style='margin: 0; color: #1a5276; font-size: 1.25em; font-weight: 800; letter-spacing: 0.5px;'>Orchid AI Insights & Analysis</h4>
            </div>
            <div style='color: #2c3e50; font-size: 1.08em; line-height: 1.75; padding-left: 5px;'>
                {formatted_result}
            </div>
            <div style='margin-top: 20px; padding-top: 12px; border-top: 1px solid #f2f4f4; font-size: 0.85em; color: #95a5a6; display: flex; align-items: center; gap: 6px;'>
                🧬 <span>Mined using Advanced NLP Index Ranking Engine • Context sourced from verified academic database.</span>
            </div>
        </div>
        """
    except Exception as e:
        return f"""
        <div style='padding:20px; background-color:#FDEDEC; border-left:6px solid #E74C3C; border-radius:12px; color:#C0392B; font-weight:bold;'>
            ⚠️ RAG Execution Fault: {str(e)}
        </div>
        """



## Step 7: Gradio GUI Web Interface Launch
This final section mounts our localized cascading style sheets (CSS), declares the interface structure mapping 4 core navigation panes (Tabs), hooks callbacks to components, and initializes the public link.

In [ ]:
# CUSTOM CSS FOR MODERN UI/UX
custom_css = """
.header-banner {
    background: linear-gradient(135deg, #1b5e20, #1abc9c);
    color: white;
    padding: 25px;
    border-radius: 16px;
    text-align: center;
    box-shadow: 0 4px 15px rgba(0,0,0,0.06);
    margin-bottom: 25px;
}
.header-banner h1 { margin: 0; font-size: 2.4em; font-weight: 800; }
.header-banner p { margin: 6px 0 0 0; font-size: 1.15em; opacity: 0.9; }

.dash-card {
    background: white;
    border-radius: 16px;
    padding: 20px;
    box-shadow: 0 4px 20px rgba(0,0,0,0.03);
    text-align: center;
    border: 1px solid #eaeded;
    transition: all 0.25s ease;
}
.dash-card:hover { transform: translateY(-4px); box-shadow: 0 8px 25px rgba(0,0,0,0.06); }
.card-icon { font-size: 42px; margin-bottom: 8px; }
.card-title { color: #7f8c8d; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1.2px; font-weight: 600; }
.card-value { color: #2c3e50; font-size: 2.2em; font-weight: 800; margin: 8px 0; }
.status-optimal { color: #27ae60; font-weight: bold; }
.status-warning { color: #e74c3c; font-weight: bold; }

.section-header {
    text-align: center;
    margin-bottom: 25px;
    padding: 10px;
}
.section-header h3 { color: #1a5276; font-size: 1.5em; font-weight: 800; margin: 0 0 5px 0; }
.section-header p { color: #7f8c8d; margin: 0; font-size: 1em; }
"""

# MAIN INTERFACE (MODERNIZED UI ARCHITECTURE)
with gr.Blocks(css=custom_css, theme=gr.themes.Default(primary_hue="emerald", secondary_hue="teal")) as app:

    gr.HTML("""
        <div class="header-banner">
            <h1>🌿 OnlyPlants Care Hub</h1>
            <p>Cloud Computing Lab Final Project • Real-Time Orchid Analytics</p>
        </div>
    """)

    with gr.Tabs():

        with gr.Tab("📊 Live Dashboard", id=1):
            gr.HTML("""
                <div class="section-header">
                    <h3>🏢 Greenhouse Climate Metrics</h3>
                    <p>Current ecosystem snapshot fetched from synchronized cloud data states.</p>
                </div>
            """)
            out4 = gr.HTML(value=update_dashboard())
            btn4 = gr.Button("🔄 Refresh Dashboard View", variant="primary", size="lg")
            btn4.click(fn=update_dashboard, inputs=None, outputs=out4)

        with gr.Tab("📡 IoT Telemetry", id=2):
            gr.HTML("""
                <div class="section-header">
                    <h3>📡 Cloud Telemetry Gateway</h3>
                    <p>Request data updates from the proxy node server and trigger cloud data synchronization.</p>
                </div>
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    feed = gr.Dropdown(choices=["humidity", "soil", "temperature", "json"], label="Select Feed Channel", value="json")
                    limit = gr.Slider(1, 50, 10, label="Historical Samples Buffer Limit")
                    btn2 = gr.Button("⬇️ Fetch & Sync Gateway", variant="primary")
                with gr.Column(scale=2):
                    out2 = gr.HTML(label="Cloud Sync Output Status")
            btn2.click(fn=fetch_iot_data, inputs=[feed, limit], outputs=out2)

        with gr.Tab("🔍 Ask the Agronomist (RAG)", id=3):
            gr.HTML("""
                <div class="section-header">
                    <h3>🧠 Retrieval-Augmented Agronomist Intelligence</h3>
                    <p>Submit multi-word questions to cross-reference text-chunks with 5 core orchid research papers.</p>
                </div>
            """)
            with gr.Row():
                with gr.Column(scale=3):
                    q = gr.Textbox(label="Enter your research query", placeholder="e.g., How do sensors monitor greenhouse humidity and temperature?")
                with gr.Column(scale=1):
                    s = gr.Slider(1, 5, 3, step=1, label="Max Document Contexts")
            btn3 = gr.Button("🚀 Execute Semantic Search", variant="primary")
            out3 = gr.HTML()
            btn3.click(fn=handle_rag_query, inputs=[q, s], outputs=out3)

        with gr.Tab("📷 AI Scan", id=4):
            gr.HTML("""
                <div class="section-header">
                    <h3>📸 Computer Vision Diagnosis</h3>
                    <p>Upload a high-resolution image to cross-examine cellular tissue for fungal rots or stress.</p>
                </div>
            """)
            with gr.Row():
                with gr.Column():
                    img = gr.Image(label="Source Image Input", type="filepath")
                    btn = gr.Button("🔍 Evaluate Tissue Integrity", variant="primary")
                with gr.Column():
                    out = gr.HTML(label="Visual Classifier Response")
            btn.click(fn=analyze_plant_image_html, inputs=img, outputs=out)

app.launch(share=True)